# **User's Guide for the Wonderland Elections Database**
### This Jupyter Notebook uses Python and SQL Magic to provide a user interface for inserting, deleting, modifying, and querying data in the **'wonderland_elections_project'** database.

## **Important Prerequisites**
* Please make sure that this Jupyter Notebook file is located in the same folder as the SQL scripts.
* Please make sure to run the files `createAll.sql` and `loadAll.sql` in `MySQL Workbench` **BEFORE** you use this Jupyter Notebook.
(**Note:** Instructions on how to create the relations and load the data for the **'wonderland_elections_project'** database using the SQL files can be seen in the `README` document)
* Please make sure to run the **MySQL to Python Connector Cell**, **Database Configuration Cell**, and **SQL Magic Configuration Cell** in sequential order before running the cells in Part A and Part B.
* **Important Note For The Database Configuration Cell:** Please read the instructions regarding modification of the user, password, and port in the config.

## **This Jupyter Notebook includes two parts...**
## Part A: Activities
### This section provides a user interface for Elections Staff and Folks to do various tasks
* **A.1:** Allows a clerk in the Elections Staff to create a new poll for the Wonderland Elections
* **A.2:** Allows a folk to register to vote at a voting center of their choice
* **A.3:** Allows a clerk to modify the availability period of a poll (*Note: poll must not have ballots cast to it)
* **A.4:** Allows a folk to cast a ballot; utilizes a SQL transaction to confirm the validity of the voting registration, time cast, and prevent double voting
* **A.5:** Allows an Elections Staff member to remove a folk from the **'wonderland_elections_project'** and all their associated information

## Part B: Queries/Reports
### This section provides a user interface for to view various reports regarding the Wonderland Elections. Also includes sections for user inputs for some queries to view personalized reports.
* **B.1:** List the name, city, and email (any single email suffices) of all folks.
* **B.2:** List the city, state, and the number of residents of each city in Wonderland
(skip cities with no residents) in decreasing order of number of residents.
* **B.3:** List each center together with its number of currently registered folks (include
states with no inhabited places) in increasing alphabetical order of their
zipcode.
* **B.4:** Find the distinct identifiers and names of folks registered at a given voting
center within a given time period.
* **B.5:** Find for a given month, the number of unique registrations at any voting
center which is within 3 miles from the center of Megapolis, except for voting
centers in a given (exclusion) list of voting centers.
* **B.6:** Find the most popular voting center(s) (in terms of total number of
registrations) in a given time period among those in a given city.
* **B.7:** Find the unique folk that have valid registrations with every voting center on a
given state.
* **B.8:** Find folks that registered at a voting center that is farther away than the voting
center closest to their residence (break ties alphabetically by center’s
acronym).
* **B.9:** Write a SQL function that returns the acronym of the voting center closest to
the residence of a given folk among those whose operating period(s) contain a
given date (return NULL if no such center exists; break ties at alphabetically
by acronym).
* **B.10:** For a given poll, construct a cross-tabulation of voting centers (acronym) as
rows, unique poll answers (options) as columns, and cells indicating number
of cast ballots at the row’s center with the column’s option.

## **Instructions**
* **Please read the instructions specified under each problem.**
* **Run each cell sequentially. Can do ``Shift`` + `Enter`**
* **For 'User Interface Cells' in Part A and 'Input Cells' in Part B, type your inputs into the text box when prompted and hit `Enter`.**

## **End Of User's Guide for the Wonderland Elections Database...Start running the cells below**

# MySQL to Python Connector Cell
## Run the cell below **first** to install the MySQL to Python Connector

In [ ]:
!pip install mysql-connector-python

# Database Configuration Cell
## Run the cell below **second** to run the configuration to connect to the `wonderland_elections_project` database
**Important Note:**
Before running the cell below, edit **`config`** with credentials to match your local MySQL Environment. Also for ease, please keep the database name to be `wonderland_elections_project`.

* **`user`**: Change this to match your MySQL username (default is `'root'`).
* **`password`**: Change this to match your local MySQL password
* **`port`**: Change this to match your local MySQL port (the default port is `3306`).

In [ ]:
# Database Server Connection Configuration
import mysql.connector

config = {
   'user': 'root',
   'password': 'Nighthawk27!',
   'host': 'localhost',
   'port': 3306,
   'database': 'wonderland_elections_project',
   'raise_on_warnings': True                    
}

print("Configuration successfully loaded. You are now connected to the database.")

# SQL Magic Configuration Cell
## Run the cell below **third** to install and load SQL Magic

In [ ]:
# Install the libraries
!pip install ipython-sql sqlalchemy

# Load the extension
%load_ext sql

# Create a connection string to the database using config
connection_string = f"mysql+mysqlconnector://{config['user']}:{config['password']}@{config['host']}/{config['database']}"

# Connect SQL Magic to the database
%sql $connection_string

# Used to make sure prettytable version works
!pip install --force-reinstall prettytable==2.5.0

# **Part A: Activities**

# A.1: A clerk creating a new poll

**Description:** 
This section below will allow a clerk to create a new poll in the wonderland_elections_project database. To do this, I created a **Setup Cell** with the Python function `create_poll(poll_name_id, question_text, start_datetime_availability, end_datetime_availability)`. This function communicates with the MySQL database server and executes a SQL query to insert a tuple into the Poll relation with the arguments specified by the clerk. I also created a **User Interface Cell** that the clerk will use to enter information for the new poll. Lastly, the **Verification Cell** can be used to verify that the poll is in the database by querying using SQL Magic.

**Tip For Tester:** Can use `1357900000000000` as elections staff clerk folk_id

**Instructions:**
1. Run the **Setup Cell**; this defines the `create_poll()` function.
2. Run the **User Interface Cell**; this will ask the user to input the information for the poll. Enter the information when prompted.
3. Run the **Verification Cell** only if the result from step 2 is that the Poll was created successfully. Here, enter the Poll Name ID from step 2 to verify that the new Poll is in the database.

## A.1: Setup Cell

In [ ]:
# A.1: Setup Cell
def create_poll(staff_folk_id, poll_name_id, question_text, start_datetime_availability, end_datetime_availability):
    configuration_connection = None
    cursor = None
    try:
        # Connect to the MySQL Database
        configuration_connection = mysql.connector.connect(**config)
        cursor = configuration_connection.cursor()

        # Ask user for their folk_id to make sure they are an elections staff member
        staff_verification_query = "SELECT staff_role FROM Elections_Staff WHERE folk_id = %s"
        cursor.execute(staff_verification_query, (staff_folk_id,))
        staff_verification_result = cursor.fetchone()

        # If the user is not a staff member or a clerk, end the function
        if not staff_verification_result or staff_verification_result[0] != 'Clerk':
            print(f"Access Denied: Cannot use function as '{staff_folk_id}' is not a member of the Elections Staff or not a Clerk.")
            return
        
        # Define the query that will create a new Poll tuple with the inputted information
        insert_poll_query = ("INSERT INTO Poll(poll_name_id, question_text, start_datetime_availability, end_datetime_availability) VALUES (%s, %s, %s, %s)")
        
        # Execute and commit the query
        cursor.execute(insert_poll_query, (poll_name_id, question_text, start_datetime_availability, end_datetime_availability))
        configuration_connection.commit()

        # Print this statement to let the user know the Poll tuple was created
        print(f"Poll '{poll_name_id}' created successfully.")

    # If there was an error when creating the new Poll tuple, send an error message
    except Error as e:
        print(f"Poll was not created. There is an error: {e}")

    # Close the cursor and connection to the database server
    finally:
        cursor.close()
        configuration_connection.close()

## A.1: User Interface Cell

**Note for User Interface Cell Inputs:** 
1. `new_poll_name_id_input` must be 4 alphanumeric characters [A-Z0-9] (ex., `NP01`)
2. `new_start_datetime_availability_input` and `new_end_datetime_availability_input` must be in `YYYY-MM-DD HH:MM:SS` format (ex., `2025-12-01 09:30:00`).

In [ ]:
# A.1: User Interface Cell
print("--- Input the information for the new Poll below ---\n")

# Ask to user to input their Elections Staff ID
elections_staff_id_input = input("Enter your Elections Staff ID(16 digits): ")

# Asks the user to input the information for the new Poll
new_poll_name_id_input = input("Enter new Poll Name ID(must be 4 alphanumeric characters): ")
new_question_text_input = input("Enter the question for the new Poll: ")
new_start_datetime_availability_input = input("Enter Start Datetime Availability(YYYY-MM-DD HH:MM:SS): ")
new_end_datetime_availability_input = input("Enter End Datetime Availability(YYYY-MM-DD HH:MM:SS): ")

print("\nCreating new poll...sending information to the database.")

# Call the create_poll() function using the user's inputs as the arguments
create_poll(elections_staff_id_input, new_poll_name_id_input, new_question_text_input, new_start_datetime_availability_input, new_end_datetime_availability_input)

## A.1: Verification Cell
**Note for Verification Cell:** 
Only run this cell if the result of the **User Interface Cell** is that the Poll was successfully created. Also, please input the same exact Poll Name ID from the recently run User Interface Cell when prompted.

In [ ]:
# A.1: Verification Cell

# Ask user for the Poll Name ID
print("Here you can verify if the newly created Poll has been saved in the database.\n")
new_poll_name_id_input = input("Enter the Poll Name ID of the newly created Poll: ")

print(f"\nSearching database for the Poll: {new_poll_name_id_input}")

print("Here is the Poll you created from the Poll relation in the database:\n")

# Use SQL Magic to query the database and display the Poll tuple
%sql SELECT * FROM Poll WHERE poll_name_id = :new_poll_name_id_input;

# A.2: A folk registering to vote at a center

**Description:** 
This section below will allow a folk to create a registration to vote at a designated voting center in Wonderland. It will save the voting registration in the wonderland_elections_project database. To do this, I created a **Setup Cell** with the Python function `register_folk(folk_id, center_id, desired_voting_date)`. This function communicates with the MySQL database server and executes a SQL query to insert a tuple into the Voting_Registration relation with the arguments specified by the folk. I also created a **User Interface Cell** that the folk will use to enter information for their registration. Lastly, the **Verification Cell** can be used to verify that the voting registration is in the database by querying using SQL Magic.

**Instructions:**
1. Run the **Setup Cell**; this defines the `register_folk()` function.
2. Run the **User Interface Cell**; this will ask the user to input the information for the voting registration. Enter the information when prompted.
3. Run the **Verification Cell** only if the result from step 2 is that the Voting_Registration was created successfully. Here, enter the Poll Name ID from step 2 to verify that the new Poll is in the database.

## A.2: Setup Cell

In [ ]:
# A.2: Setup Cell
def register_folk(folk_id, center_id, desired_voting_date):
    configuration_connection = None
    cursor = None
    try:    
        # Connect to the MySQL Database
        configuration_connection = mysql.connector.connect(**config)
        cursor = configuration_connection.cursor()
        
        # Define the query that will create a new Voting_Registration tuple with the inputted information
        insert_registration_query = ("INSERT INTO Voting_Registration(folk_id, center_id, desired_voting_date) VALUES (%s, %s, %s)")
        
        # Execute and commit the query
        cursor.execute(insert_registration_query, (folk_id, center_id, desired_voting_date))
        configuration_connection.commit()

        # Gets the registration_id of the newly created tuple
        new_registration_id = cursor.lastrowid
        
        # Print this statement to let the user know the Voting_Registration tuple was created
        print(f"Registration for Folk '{folk_id}' at Voting Center '{center_id}' was created successfully.")
        print(f"Here is your Registration ID: {new_registration_id}")

    except Error as e:
        print(f"Voting Registration was not created. There is an error: {e}")
    
    # Close the cursor and connection to the database server
    finally:
        cursor.close()
        configuration_connection.close()

## A.2: User Interface Cell

**Note for User Interface Cell Inputs:** 
1. `new_folk_id_input` must be 16 digits and must already exist in the Folks Relation (ex., `1357900000000010`)
2. `new_center_id_input` must be an integer of a place_id that already exists in the Voting_Center Relation (ex., `5` for Voting Center `DAMA`).
3. `new_desired_voting_date_input` must be in `YYYY-MM-DD` format and must be within a valid operating period of the Voting Center (ex., `2026-02-14` for Voting Center `DAMA`).

In [ ]:
# A.2: User Interface Cell
print("--- Input the information for the new Voting_Registration below ---\n")

# Ask the user to input the information for their new Voting_Registration tuple
new_folk_id_input = input("Enter your Folk ID(must be 16 digits): ")
new_center_id_input = input("Enter ID of your desired Voting Center(must be integer, ex., 5): ")
new_desired_voting_date_input = input("Enter your Desired Voting Date(YYYY-MM-DD): ")

print("\nCreating new registration...sending information to the database.")

# Call the register_folk() function using the user's inputs as the arguments
register_folk(new_folk_id_input, new_center_id_input, new_desired_voting_date_input)

## A.2: Verification Cell
**Note for Verification Cell:** 
Only run this cell if the result of the **User Interface Cell** is that the Voting_Registration was successfully created. Also, please input the same exact Registration ID from the recently run User Interface Cell when prompted.

In [ ]:
# A.2: Verification Cell

# Ask user for the Registration ID
print("Here you can verify if the newly created Voting Registration has been saved in the database.\n")
new_registration_id_input = input("Enter the Registration ID of the newly created Voting_Registration(displayed in A.2 User Interface Cell): ")

print(f"\nSearching database for the Voting Registration: {new_registration_id_input}")

print("Here is the registration you created from the Voting_Registration relation in the database:\n")

# Use SQL Magic to query the database and display the Voting_Registration tuple
%sql SELECT * FROM Voting_Registration WHERE registration_id = :new_registration_id_input;

# A.3: A clerk modifying the availability period of a current poll

**Description:** 
This section below will allow a clerk to modify the availability period of a Poll in Wonderland. It will save the modification to the Poll in the wonderland_elections_project database. To do this, I created a **Setup Cell** with the Python function `modify_poll_availability_period(poll_name_id, start_datetime_availability, end_datetime_availability)`. This function communicates with the MySQL database server and executes a SQL query to modify a tuple in the Poll relation with the arguments specified by the folk. I also created a **User Interface Cell** that the clerk will use to enter the new availability period for a specified Poll. Lastly, the **Verification Cell** can be used to verify that the poll has been modified in the database by querying using SQL Magic.

**Note:** A Poll's availability period may only be modified if there are no cast ballots for the poll.

**Tip For Tester:** Can use `1357900000000000` as elections staff clerk folk_id

**Instructions:**
1. Run the **Setup Cell**; this defines the `modify_poll_availability_period()` function.
2. Run the **User Interface Cell**; this will ask the user to input the ID of the Poll and the new availability period. Enter the information when prompted.
3. Run the **Verification Cell**; this will allow the user to check if the availability period of the poll was successfully modified in the database.

## A.3: Setup Cell

In [ ]:
# A.3: Setup Cell
def modify_poll_availability_period(staff_folk_id, poll_name_id, start_datetime_availability, end_datetime_availability):
    configuration_connection = None
    cursor = None
    try:
        # Connect to the MySQL Database
        configuration_connection = mysql.connector.connect(**config)
        cursor = configuration_connection.cursor()

        # Ask user for their folk_id to make sure they are an elections staff member
        staff_verification_query = "SELECT staff_role FROM Elections_Staff WHERE folk_id = %s"
        cursor.execute(staff_verification_query, (staff_folk_id,))
        staff_verification_result = cursor.fetchone()

        # If the user is not a staff member or a clerk, end the function
        if not staff_verification_result or staff_verification_result[0] != 'Clerk':
            print(f"Access Denied: Cannot use function as '{staff_folk_id}' is not a member of the Elections Staff or not a Clerk.")
            return
        
        # Define the query that will modify the Poll tuple's availability period with the user inputted datetimes
        poll_availability_period_modification_query = ("UPDATE Poll SET start_datetime_availability = %s, end_datetime_availability = %s WHERE poll_name_id = %s")
        
        # Execute and commit the query
        cursor.execute(poll_availability_period_modification_query, (start_datetime_availability, end_datetime_availability, poll_name_id))
        configuration_connection.commit()
        
        print(f"The availability period for Poll '{poll_name_id}' was updated successfully.")

    except Error as e:
        print(f"Poll was not modified. There is an error: {e}")
        print("**Note:** A Poll's availability period may only be modified if there are no cast ballots for the poll.")
    
    # Close the cursor and connection to the database server
    finally:
        cursor.close()
        configuration_connection.close()

## A.3: User Interface Cell

**Note for User Interface Cell Inputs:** 
1. `elections_staff_id_input` must exist in the `Elections_Staff` relation and must be a 16-digit folk_id
2. `poll_name_id_input` must already exist in the database and must be 4 alphanumeric characters [A-Z0-9] (ex., `RARA`)
3. `new_start_datetime_input` and `new_end_datetime_input` must be in `YYYY-MM-DD HH:MM:SS` format (ex., `2025-12-01 09:30:00`).

In [ ]:
# A.3: User Interface Cell
print("--- Input the new availability period for an existing Poll below ---\n")

# Ask to user to input their Elections Staff ID
elections_staff_id_input = input("Enter your Elections Staff ID(16 digits): ")

# Asks the user to input the Poll Name ID and it's new availability period
poll_name_id_input = input("Enter the ID of the Poll you want to modify(ex., RARA): ")
new_start_datetime_input = input("Enter the New Start Datetime(YYYY-MM-DD HH:MM:SS): ")
new_end_datetime_input = input("Enter the New End Datetime(YYYY-MM-DD HH:MM:SS): ")

print("\nModifying the availability period of the Poll...sending information to the database.")

# Call the modify_poll_availability_period() function using the user's inputs as the arguments
modify_poll_availability_period(elections_staff_id_input, poll_name_id_input, new_start_datetime_input, new_end_datetime_input)

## A.3: Verification Cell
**Note for Verification Cell:** 
Please input the same exact `poll_name_id` from the recently run User Interface Cell when prompted. This will allow you to view the information of the Poll you are attempting to modify.

In [ ]:
# A.3: Verification Cell

# Ask user for the poll_name_id
print("Here you can verify if the Poll's availability period modification has been saved in the database.\n")
poll_name_id_input = input("Enter the Poll Name ID to view: ")

print(f"\nSearching database for the Poll: {poll_name_id_input}")

print("Here is the Poll you modified from the Poll relation in the database:\n")

# Use SQL Magic to query the database and display the Poll tuple
%sql SELECT * FROM Poll WHERE poll_name_id = :poll_name_id_input;

# A.4: A voter casting a ballot while confirming a valid voting registration

**Description:** 
This section below will allow a folk to cast a ballot for a Poll in Wonderland. It will save the created Ballot in the wonderland_elections_project database. To do this, I created a **Setup Cell** with the Python function `transaction_cast_ballot(voter_choice, cast_datetime, registration_id, poll_name_id)`. This function communicates with the MySQL database server and executes a SQL query to insert a tuple in the Ballot relation with the arguments specified by the folk. This function implements a transaction in SQL to ensure data integrity in the database. This is to rollback the insertion of a ballot if it attempted to be submitted outside of the operating period of the voting center, or if the associated registration is invalid. It also prevents double-voting. I also created a **User Interface Cell** that the folk will use to enter the information needed to submit the ballot. Lastly, the **Verification Cell** can be used to verify that the Ballot has been inserted in the database by querying using SQL Magic.

**Note:** A folk can only cast a ballot for a poll on the date and center of one of their valid
registrations, and during the operating hours of that center on that date.

**Instructions:**
1. Run the **Setup Cell**; this defines the `transaction_cast_ballot()` function.
2. Run the **User Interface Cell**; this will ask the user to input the ID of the Poll and the new availability period. Enter the information when prompted.
3. Run the **Verification Cell**; this will allow the user to check if the availability period of the poll was successfully modified in the database.

## A.4: Setup Cell

In [ ]:
# A.4: Setup Cell
def transaction_cast_ballot(voter_choice, cast_datetime, registration_id, poll_name_id):
    configuration_connection = None
    cursor = None
    try:
        # Connect to the MySQL Database
        configuration_connection = mysql.connector.connect(**config)
        cursor = configuration_connection.cursor()
        
        print("Starting Transaction To Create And Insert Ballot...")
        
        # Use SQL to Set Isolation Level to SERIALIZABLE; prevents double-voting
        cursor.execute("SET TRANSACTION ISOLATION LEVEL SERIALIZABLE")
        
        # Begin the SQL Transaction
        configuration_connection.start_transaction()
        
        # Define query to insert a tuple into the Ballot relation; 
        # Note that folk_id and center_id are automatically set via ballot_insert_logic trigger
        ballot_insert_query = ("INSERT INTO Ballot(voter_choice, cast_datetime, registration_id, poll_name_id) VALUES (%s, %s, %s, %s)")
        
        # Execute query
        cursor.execute(ballot_insert_query, (voter_choice, cast_datetime, registration_id, poll_name_id))

        # Gets the ballot_id of the newly created tuple
        new_ballot_id = cursor.lastrowid
        
        # Commit query
        configuration_connection.commit()
        print("The Ballot was successfully cast and saved in the database! Thank you for voting!")
        print(f"Here is the ID of your Ballot: {new_ballot_id}")
        
    except Error as e:
        # Commit a rollback if any error occurred during the transaction
        if configuration_connection:
            configuration_connection.rollback()
        print(f"Ballot was not cast. The transaction has been rolled-back. There is an error: {e}")
        
    # Close the cursor and connection to the database server
    finally:
        cursor.close()
        configuration_connection.close()

## A.4: User Interface Cell

**Note for User Interface Cell Inputs:** 
1. `voter_choice` value must be `YES`, `NO`, or `ABSTAIN`
2. `cast_datetime` must be datetime format`(YYYY-MM-DD HH:MM:SS)`
3. `registration_id` must exist in the Voting_Registration relation and must be valid; input must be an integer (ex., `51`)
4. `poll_name_id` must exist in the Poll relation and must be 4 alphanumeric characters [A-Z0-9] (ex., `RARA`)

In [ ]:
# A.4: User Interface Cell
print("--- Input the details of your Ballot below ---\n")

# Asks the user to input 
registration_id_input = input("Enter Registration ID(ex., 51): ")
poll_name_id_input = input("Enter Poll Name ID(ex., RARA): ")
voter_choice_input = input("Enter Your Vote For The Poll(YES, NO, ABSTAIN): ")
cast_datetime_input = input("Enter The Datetime Of Ballot Cast(YYYY-MM-DD HH:MM:SS): ")

print("\nCasting Your Ballot...sending information to the database.")

# Call the transaction_cast_ballot() function using the user's inputs as the arguments
transaction_cast_ballot(voter_choice_input, cast_datetime_input, registration_id_input, poll_name_id_input)

## A.4: Verification Cell
**Note for Verification Cell:** 
Please input the same exact `ballot_id` from the recently run User Interface Cell when prompted. This will allow you to view the information of the Ballot recently cast.

In [ ]:
# A.4: Verification Cell

# Ask user for the Ballot ID (from the output above)
print("Here you can verify if your recently cast Ballot has been saved in the database.\n")
ballot_id_input = input("Enter the new Ballot ID to view: ")

print(f"\nSearching database for Ballot ID: {ballot_id_input}...\n")

print("Here is the Ballot you cast in the Ballot relation in the database:\n")

# Use SQL Magic to query the database and display the Poll tuple
%sql SELECT * FROM Ballot WHERE ballot_id = :ballot_id_input;

# A.5: A staff removing a folk (and all their associated information)

**Description:** 
This section below will allow a staff in Wonderland to remove a folk and all their information. It will delete the `Folks` tuple and their associated `Folks_Email` and `Voting_Registrations` tuples in the wonderland_elections_project database. To do this, I created a **Setup Cell** with the Python function `staff_removal(staff_folk_id, folk_to_be_deleted_id)`. This function communicates with the MySQL database server and executes a SQL query to delete a specified folk from the Folks relation. I also created a **User Interface Cell** that the staff can use to enter the ID of the folk they want to delete. Lastly, the **Verification Cell** can be used to verify that the folk has been removed from the database by querying using SQL Magic.

**Note:** The function will ask for a folk_id to verify that the user of this function is a staff member in the Elections_Staff relation. 
### NOTE: The database will block the deletion of the folk if they have cast a ballot for a poll.

**Tip For Tester:** Can use `1357900000000000` as elections staff folk_id and `1357900000000015` as the folk_id to be deleted.

**Instructions:**
1. Run the **Setup Cell**; this defines the `staff_removal()` function.
2. Run the **User Interface Cell**; this will ask to enter your elections staff folk_id, and the folk_id of the folk you want to delete from the database.
3. Run the **Verification Cell**; this will allow the staff member to check if the folk has been successfully deleted.

## A.5: Setup Cell

In [ ]:
# A.5: Setup Cell
def folk_removal(staff_folk_id, folk_to_be_deleted_id):
    configuration_connection = None
    cursor = None
    try:
        # Connect to the MySQL Database
        configuration_connection = mysql.connector.connect(**config)
        cursor = configuration_connection.cursor()

        # Ask user for their folk_id to make sure they are an elections staff member
        staff_verification_query = "SELECT staff_role FROM Elections_Staff WHERE folk_id = %s"
        cursor.execute(staff_verification_query, (staff_folk_id,))
        staff_verification_result = cursor.fetchone()

        # If the user is not a staff member, end the function
        if not staff_verification_result:
            print(f"Access Denied: Cannot use function as '{staff_folk_id}' is not a member of the Elections Staff.")
            return

        # If verified, define query to delete specified folk from the Folks relation
        folk_deletion_query = ("DELETE FROM Folks WHERE folk_id = %s")

        # Execute and commit the query
        cursor.execute(folk_deletion_query, (folk_to_be_deleted_id,))
        configuration_connection.commit()

        print(f"The folk and all of their associated information was successfully deleted.")

    except Error as e:
        print(f"Folk was not deleted from the database. There is an error: {e}")
        print("This can be due to either the folk having a cast ballot, or the folk does not exist in the database.")
    
    # Close the cursor and connection to the database server
    finally:
        cursor.close()
        configuration_connection.close()

## A.5: User Interface Cell

**Note for User Interface Cell Inputs:** 
1. `elections_staff_id_input` must exist in the `Elections_Staff` relation and must be a 16-digit folk_id
2. `folk_to_be_deleted_input` must exist in the `Folks` relation and must be a 16-digit folk_id

In [ ]:
# A.5: User Interface Cell
print("--- Input your Elections Staff ID and the Folk ID to be deleted below ---\n")

# Ask to user to input their Elections Staff ID
elections_staff_id_input = input("Enter your Elections Staff ID(16 digits): ")

# Ask the user to input the ID of the Folk they want to delete
folk_to_be_deleted_input = input("Enter the Folk ID to remove(16 digits): ")

print(f"\nAttempting to remove the Folk {folk_to_be_deleted_input}...please wait.")

# Call the folk_removal() function using the user's inputs as the arguments
folk_removal(elections_staff_id_input, folk_to_be_deleted_input)

## A.5: Verification Cell
**Note for Verification Cell:** 
Please input the same exact `folk_id` from the recently run User Interface Cell when prompted. This should show up empty.

In [ ]:
# A.5: Verification Cell

# Ask user for the folk_id to be deleted from the user interface cell
print("Here you can verify if the Folk still exists in the database.\n")
deleted_folk_id = input("Enter folk_id that was recently deleted: ")

print(f"\nSearching database for the Folk: {deleted_folk_id}")

print("If the table below is empty, that means the folk no longer exists in the database:\n")

# Use SQL Magic to query the database and display the Poll tuple
%sql SELECT * FROM Folks WHERE folk_id = :deleted_folk_id;

# **Part B: Queries/Reports**

# B.1: List the name, city, and email (any single email suffices) of all folks.
**--- Query/Report B.1: List the name, city, and email (any single email suffices) of all folks. ---**

Run the cell below, it will query the database using SQL Magic to retrieve the report.

In [ ]:
%%sql 
SELECT Folks.first_name, Folks.last_name, Place.city, MIN(Folks_Email.email) AS email
FROM Folks, Place, Folks_Email
WHERE Folks.place_id = Place.place_id AND Folks.folk_id = Folks_Email.folk_id
GROUP BY Folks.folk_id, Folks.first_name, Folks.last_name, Place.city;

# B.2: List the city, state, and the number of residents of each city in Wonderland (skip cities with no residents) in decreasing order of number of residents.
**--- Query/Report B.2: List the city, state, and the number of residents of each city in Wonderland
(skip cities with no residents) in decreasing order of number of residents. ---**

Run the cell below, it will query the database using SQL Magic to retrieve the report.

In [ ]:
%%sql
SELECT Place.city, Place.state, COUNT(Folks.folk_id) AS number_of_residents
FROM Folks, Residence, Place
WHERE Folks.place_id = Residence.place_id AND Residence.place_id = Place.place_id
GROUP BY Place.city, Place.state
HAVING number_of_residents > 0
ORDER BY number_of_residents DESC;

# B.3: List each center together with its number of currently registered folks (include states with no inhabited places) in increasing alphabetical order of their zipcode.
**--- Query/Report B.3: List each center together with its number of currently registered folks (include states with no inhabited places) in increasing alphabetical order of their zipcode. ---**

Run the cell below, it will query the database using SQL Magic to retrieve the report.

In [ ]:
%%sql
SELECT Voting_Center.acronym, Place.zip_code, COUNT(Voting_Registration.registration_id) AS num_registered_folks
FROM Voting_Center
INNER JOIN Place ON Voting_Center.place_id = Place.place_id
LEFT OUTER JOIN Voting_Registration ON Voting_Center.place_id = Voting_Registration.center_id
GROUP BY Voting_Center.place_id, Voting_Center.acronym, Place.zip_code
ORDER BY Place.zip_code ASC;

#  B.4: Find the distinct identifiers and names of folks registered at a given voting center within a given time period.
**--- Query/Report B.4: Find the distinct identifiers and names of folks registered at a given voting
center within a given time period. ---**

**Instructions:**
1. Run the **B.4: Input Cell** first; this is where you define the voting center and time period
2. Run the **B.4: SQL Cell** second; this will query the database using your inputs

## B.4: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_center_id` must exist in the `Voting_Center` relation and an integer(ex., `5` for `DAMA`)
2. `inputted_start_date` and `inputted_end_date` must be a date format`(YYYY-MM-DD)`

In [ ]:
# Input Cell: Get user inputs for Query/Report B.4
print("--- Get User Inputs For Query/Report B.4 ---\n")

print("Please input the desired voting center place_id and time period below.\n")

# Get inputs from the user
inputted_center_id = input("Enter the ID of the Voting Center(ex., 5 for DAMA): ")
inputted_start_date = input("Enter the Start Date for the report(YYYY-MM-DD): ")
inputted_end_date = input("Enter the End Date for the report(YYYY-MM-DD): ")

print("The variables have been successfully set with your inputs. Please run the SQL Cell below to view the report.")

## B.4: SQL Cell

In [ ]:
%%sql
SELECT DISTINCT Folks.folk_id, Folks.first_name, Folks.last_name
FROM Folks, Voting_Registration
WHERE Folks.folk_id = Voting_Registration.folk_id AND Voting_Registration.center_id = :inputted_center_id 
	AND Voting_Registration.desired_voting_date BETWEEN :inputted_start_date AND :inputted_end_date;

#  B.5: Find for a given month, the number of unique registrations at any voting center which is within 3 miles from the center of Megapolis, except for voting centers in a given (exclusion) list of voting centers.
**--- Query/Report B.5: Find for a given month, the number of unique registrations at any voting
center which is within 3 miles from the center of Megapolis, except for voting
centers in a given (exclusion) list of voting centers. ---**

**NOTE: MEGAPOLIS COORDINATES ARE (0,0)**

**Instructions:**
1. Run the **B.5: Input Cell** first; this is where you define the month, year, and voting center
2. Run the **B.5: SQL Cell** second; this will query the database using your inputs

## B.5: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_month` must exist be an 2-digit integer(ex., `01` for `January`)
2. `excluded_center_id` must exist in the `Voting_Center` relation and an integer(ex., `5` for `DAMA`)
3. If you do not want to exclude any voting centers, just put 1000 for `excluded_center_id` because there is not a voting center with that ID

In [ ]:
# Input Cell: Get user inputs for Query/Report B.5
print("--- Get User Inputs For Query/Report B.5 ---\n")

print("Please input the desired month, year, and voting centers to exclude below.\n")

# Get inputs from the user
inputted_month = input("Enter the Month for the report(ex., 01 for January): ")
inputted_year = input("Enter the Year for the report(ex., 2026): ")
excluded_center_id = input("Enter the Voting Center to exclude(ex., 5 for DAMA): ")

print("The variables have been successfully set with your inputs. Please run the SQL Cell below to view the report.")

## B.5: SQL Cell

In [ ]:
%%sql
SELECT COUNT(DISTINCT Voting_Registration.registration_id) AS unique_voting_registrations
FROM Voting_Registration, Voting_Center, Place
WHERE Voting_Registration.center_id = Voting_Center.place_id
    AND Voting_Center.place_id = Place.place_id
    AND MONTH(Voting_Registration.desired_voting_date) = :inputted_month 
	AND YEAR(Voting_Registration.desired_voting_date) = :inputted_year
    AND Voting_Registration.center_id != :excluded_center_id
    AND SQRT(POWER(Place.x_coordinate, 2) + POWER(Place.y_coordinate, 2)) < 3;

#  B.6: Find the most popular voting center(s) (in terms of total number of registrations) in a given time period among those in a given city.
**--- Query/Report B.6: Find the most popular voting center(s) (in terms of total number of
registrations) in a given time period among those in a given city. ---**

**Instructions:**
1. Run the **B.6: Input Cell** first; this is where you define the city, start date, and end date
2. Run the **B.6: SQL Cell** second; this will query the database using your inputs

## B.6: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_city` must be a string and must exist as a city in the Place relation(ex., `Damascus`)
2. `inputted_start_date` and `inputted_end_date` must be a date format`(YYYY-MM-DD)`

In [ ]:
# Input Cell: Get user inputs for Query/Report B.6
print("--- Get User Inputs For Query/Report B.6 ---")

print("Please input the desired month, year, and voting centers to exclude below.\n")

# Get inputs from the user
inputted_city = input("Enter City Name for the report(ex., Damascus): ")
inputted_start_date = input("Enter Start Date for the report(YYYY-MM-DD): ")
inputted_end_date = input("Enter End Date for the report(YYYY-MM-DD): ")

print("The variables have been successfully set with your inputs. Please run the SQL Cell below to view the report.")

## B.6: SQL Cell

In [ ]:
%%sql
WITH Total_Registrations_Count_Per_Voting_Center AS (
    SELECT Voting_Center.acronym, COUNT(Voting_Registration.registration_id) AS total_number_of_registrations
    FROM Voting_Center, Place, Voting_Registration
    WHERE Voting_Center.place_id = Place.place_id
      AND Voting_Center.place_id = Voting_Registration.center_id
      AND Place.city = :inputted_city
      AND Voting_Registration.desired_voting_date BETWEEN :inputted_start_date AND :inputted_end_date
    GROUP BY Voting_Center.place_id, Voting_Center.acronym
)
SELECT acronym, total_number_of_registrations
FROM Total_Registrations_Count_Per_Voting_Center
WHERE total_number_of_registrations = (SELECT MAX(total_number_of_registrations) FROM Total_Registrations_Count_Per_Voting_Center);

#  B.7: Find the unique folk that have valid registrations with every voting center on a given state.
**--- Query/Report B.7: Find the unique folk that have valid registrations with every voting center on a
given state. ---**

**Instructions:**
1. Run the **B.7: Input Cell** first; this is where you define the name of the state for the report
2. Run the **B.7: SQL Cell** second; this will query the database using your inputs

## B.7: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_state` must be a string and must exist as a state in the Place relation(ex., `Maryland`)

In [ ]:
# Input Cell: Get user input for Query/Report B.7
print("--- Get User Input For Query/Report B.7 ---")

print("Please input the desired name of the state below.\n")

# Get input from the user
inputted_state = input("Enter State Name for the report(ex., Maryland): ")

print("The variable has been successfully set with your input. Please run the SQL Cell below to view the report.")

## B.7: SQL Cell

In [ ]:
%%sql
SELECT Folks.folk_id, Folks.first_name, Folks.last_name
FROM Folks, Voting_Registration, Voting_Center, Place
WHERE Folks.folk_id = Voting_Registration.folk_id
	AND Voting_Registration.center_id = Voting_Center.place_id
    AND Voting_Center.place_id = Place.place_id
	AND Place.state = :inputted_state 
    AND Voting_Registration.validation_status = 'Valid'
GROUP BY Folks.folk_id, Folks.first_name, Folks.last_name
HAVING COUNT(DISTINCT Voting_Registration.center_id) = (
        SELECT COUNT(*)
        FROM Voting_Center
        INNER JOIN Place ON Voting_Center.place_id = Place.place_id
        WHERE Place.state = :inputted_state
    );

#  B.8: Find folks that registered at a voting center that is farther away than the voting center closest to their residence (break ties alphabetically by center’s acronym).
**--- Query/Report B.8: Find folks that registered at a voting center that is farther away than the voting
center closest to their residence (break ties alphabetically by center’s
acronym). ---**

Run the cell below, it will query the database using SQL Magic to retrieve the report.

In [ ]:
%%sql
SELECT DISTINCT Folks.folk_id, Folks.first_name, Folks.last_name, Voting_Center.acronym AS registered_voting_center,
    (
        SELECT Voting_Center.acronym
        FROM Voting_Center, Place
        WHERE Voting_Center.place_id = Place.place_id
        ORDER BY SQRT(POWER(Folk_Residence.x_coordinate - Place.x_coordinate, 2) + POWER(Folk_Residence.y_coordinate - Place.y_coordinate, 2)) ASC, Voting_Center.acronym ASC
        LIMIT 1
    ) AS closest_voting_center
FROM Folks
INNER JOIN Place AS Folk_Residence ON Folks.place_id = Folk_Residence.place_id
INNER JOIN Voting_Registration ON Folks.folk_id = Voting_Registration.folk_id
INNER JOIN Voting_Center ON Voting_Registration.center_id = Voting_Center.place_id
INNER JOIN Place AS Center_Registered_At ON Voting_Center.place_id = Center_Registered_At.place_id
WHERE SQRT(POWER(Folk_Residence.x_coordinate - Center_Registered_At.x_coordinate, 2) + POWER(Folk_Residence.y_coordinate - Center_Registered_At.y_coordinate, 2)) > (
        SELECT MIN(SQRT(POWER(Folk_Residence.x_coordinate - Place.x_coordinate, 2) + POWER(Folk_Residence.y_coordinate - Place.y_coordinate, 2)))
        FROM Voting_Center, Place
        WHERE Voting_Center.place_id = Place.place_id
    )

#  B.9: Write a SQL function that returns the acronym of the voting center closest to the residence of a given folk among those whose operating period(s) contain a given date (return NULL if no such center exists; break ties at alphabetically by acronym).
**--- Query/Report B.9: Write a SQL function that returns the acronym of the voting center closest to
the residence of a given folk among those whose operating period(s) contain a
given date (return NULL if no such center exists; break ties at alphabetically
by acronym). ---**

**Instructions:**
1. Run the **B.9: Setup Cell** first; this is where the display_closest_voting_center() function is declared
2. Run the **B.9: Input Cell** second; this is where you define the name of the state for the report
3. Run the **B.9: SQL Cell** third; this will query the database using your inputs

## B.9: Setup Cell

In [ ]:
# B.9: Setup Cell
import mysql.connector
from mysql.connector import Error

configuration_connection = None
cursor = None
try:    
    # Connect to the MySQL Database
    configuration_connection = mysql.connector.connect(**config)
    cursor = configuration_connection.cursor()

    # This try-except is used to pass the error 1305 that is sent if the function does not exist
    try:
        # Drop the function if it exists; ensures that it is not created again
        cursor.execute("DROP FUNCTION IF EXISTS display_closest_voting_center_from_folk")
    except Error as e:
        if e.errno == 1305:
            pass 
        else:
            raise e
    
    # Define the code to create SQL function
    create_closest_center_function = """
    CREATE FUNCTION display_closest_voting_center_from_folk(inputted_folk_id CHAR(16), inputted_date DATE)
    RETURNS CHAR(4)
    CONTAINS SQL READS SQL DATA
    BEGIN
        RETURN (
            SELECT Voting_Center.acronym
            FROM Voting_Center
            INNER JOIN Place AS Center ON Voting_Center.place_id = Center.place_id
            INNER JOIN Operating_Period ON Voting_Center.place_id = Operating_Period.place_id
            INNER JOIN Folks ON Folks.folk_id = inputted_folk_id
            INNER JOIN Place AS Folk_Residence ON Folks.place_id = Folk_Residence.place_id
            WHERE inputted_date BETWEEN DATE(Operating_Period.start_datetime) AND DATE(Operating_Period.end_datetime)
            ORDER BY SQRT(POWER(Folk_Residence.x_coordinate - Center.x_coordinate, 2) + POWER(Folk_Residence.y_coordinate - Center.y_coordinate, 2)) ASC,
                Voting_Center.acronym ASC
            LIMIT 1
        );
    END
    """
    
    # Execute the query
    cursor.execute(create_closest_center_function)
    print("The function display_closest_voting_center_from_folk() was successfully created.")

except Error as e:
    print(f"The function was not created. There is an error: {e}")

# Close the cursor and connection to the database server
finally:
    cursor.close()
    configuration_connection.close()

## B.9: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_folk_id` must be exist in the `Folks` relation and must be 16-digits(ex., `1357900000000000`)
2. `inputted_date` must be a date format `(YYYY-MM-DD)`

In [ ]:
# Input Cell: Get user input for Query/Report B.9
print("--- Get User Input For Query/Report B.9 ---")

print("Please input the desired folk_id and date below.\n")

# Get input from the user
inputted_folk_id = input("Enter desired folk_id for the report(ex., 1357900000000000): ")
inputted_date = input("Enter desired Date for the report(YYYY-MM-DD): ")

print("The variables have been successfully set with your inputs. Please run the SQL Cell below to view the report.")

## B.9: SQL Cell
**Note:** This cell calls the function display_closest_voting_center() from the file `queryAll.sql`

In [ ]:
%%sql
SELECT display_closest_voting_center(:inputted_folk_id, :inputted_date) AS closest_voting_center_from_folk_residence;

#  B.10: For a given poll, construct a cross-tabulation of voting centers (acronym) as rows, unique poll answers (options) as columns, and cells indicating number of cast ballots at the row’s center with the column’s option. 
**--- Query/Report B.10: For a given poll, construct a cross-tabulation of voting centers (acronym) as
rows, unique poll answers (options) as columns, and cells indicating number
of cast ballots at the row’s center with the column’s option. ---**

**Instructions:**
1. Run the **B.10: Input Cell** first; this is where you define the name of the poll for the report
2. Run the **B.10: SQL Cell** second; this will query the database using your inputs and display the cross-tabulation for the number of votes per answer, per voting center, for a given poll

## B.10: Input Cell
**Note for Input Cell Inputs:** 
1. `inputted_poll_name_id` must be exist in the `Poll` relation and must be 4 alphanumeric characters(ex., 'XP01')

In [ ]:
# Input Cell: Get user input for Query/Report B.7
print("--- Get User Input For Query/Report B.10 ---")

print("Please input the desired name of the Poll below.\n")

# Get input from the user
inputted_poll_name_id = input("Enter Poll Name for the cross-tabulation report(ex., XP01): ")

print("The variable has been successfully set with your input. Please run the SQL Cell below to view the report.")

## B.10: SQL Cell

In [ ]:
%%sql
SELECT Voting_Center.acronym AS 'Voting Center', :inputted_poll_name_id AS 'Poll Name',
    SUM(CASE WHEN Ballot.voter_choice = 'YES' THEN 1 ELSE 0 END) AS 'YES',
    SUM(CASE WHEN Ballot.voter_choice = 'NO' THEN 1 ELSE 0 END) AS 'NO',
    SUM(CASE WHEN Ballot.voter_choice = 'ABSTAIN' THEN 1 ELSE 0 END) AS 'ABSTAIN',
    COUNT(Ballot.ballot_id) AS 'Total Ballots Cast'
FROM Voting_Center
LEFT JOIN Ballot ON Voting_Center.place_id = Ballot.center_id AND Ballot.poll_name_id = :inputted_poll_name_id
GROUP BY Voting_Center.acronym;